# cane-eval Quickstart

Evaluate a LangChain agent in under 5 minutes. This notebook walks through:

1. **Build** a LangChain support agent (GPT-4o-mini)
2. **Eval** it with Claude as judge
3. **Analyze** failures with root cause analysis
4. **Mine** failures into training data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/colingfly/cane-eval/blob/main/examples/quickstart.ipynb)

## Setup

Install cane-eval and set your Anthropic API key.

In [ ]:
!pip install -q cane-eval langchain langchain-openai

In [ ]:
import os

# Anthropic key is used for Claude-as-judge scoring
# Get one at https://console.anthropic.com/
os.environ["ANTHROPIC_API_KEY"] = "sk-ant-..."  # <-- paste your key here

# OpenAI key is used for the LangChain agent itself
# Get one at https://platform.openai.com/api-keys
os.environ["OPENAI_API_KEY"] = "sk-..."  # <-- paste your key here

## Step 1: Build a LangChain agent

First, build a simple support agent using LangChain.
This is a real agent -- not a mock -- so you can see exactly how cane-eval plugs in.

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

# Build a support agent -- a simple prompt + LLM chain
system = """You are a customer support agent for an online retail store.
Answer questions about returns, shipping, payments, and account issues.
Be helpful but concise. If you don't know something, say so."""

prompt = ChatPromptTemplate.from_messages([
    ("system", system),
    ("human", "{input}"),
])

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain = prompt | llm

# Quick test
response = chain.invoke({"input": "What is your return policy?"})
print(response.content)

## Step 2: Define your test suite

Test suites define questions, expected answers, and scoring criteria.
You can load from YAML or create programmatically.

In [ ]:
from cane_eval import TestSuite

suite = TestSuite.from_dict({
    "name": "Support Agent Eval",
    "model": "claude-sonnet-4-5-20250929",
    "criteria": [
        {"key": "accuracy", "label": "Accuracy", "description": "Factual correctness vs expected answer", "weight": 40},
        {"key": "completeness", "label": "Completeness", "description": "Covers all key points", "weight": 30},
        {"key": "hallucination", "label": "Hallucination Check", "description": "No fabricated info", "weight": 30},
    ],
    "custom_rules": [
        "Never recommend competitor products",
        "Always maintain a professional tone",
    ],
    "tests": [
        {
            "question": "What is your return policy?",
            "expected_answer": "We offer a 30-day return policy for unused items in original packaging. Items must be returned with a valid receipt. Refunds are processed within 5-7 business days.",
            "tags": ["policy"],
        },
        {
            "question": "How do I reset my password?",
            "expected_answer": "Go to the login page, click Forgot Password, enter your email, check your inbox for a reset link, then create a new password. The link expires after 24 hours.",
            "tags": ["account"],
        },
        {
            "question": "Do you offer international shipping?",
            "expected_answer": "Yes, we ship to over 50 countries. International shipping takes 7-14 business days. Costs vary by destination and are calculated at checkout.",
            "tags": ["shipping"],
        },
        {
            "question": "What payment methods do you accept?",
            "expected_answer": "We accept Visa, Mastercard, American Express, PayPal, Apple Pay, and Google Pay. All transactions are encrypted with SSL.",
            "tags": ["payments"],
        },
        {
            "question": "How do I contact customer support?",
            "expected_answer": "Reach us via live chat on our website (24/7), email at support@example.com (4-hour response), or phone at 1-800-555-0199 (Mon-Fri 9am-6pm EST).",
            "tags": ["support"],
        },
    ],
})

print(f"Suite: {suite.name}")
print(f"Tests: {len(suite.tests)}")
print(f"Criteria: {[c.key for c in suite.criteria]}")

## Step 3: Run the eval

Claude judges each response against your criteria and expected answers.

In [ ]:
from cane_eval import EvalRunner

runner = EvalRunner(verbose=True)

# Wrap the LangChain chain as a callable agent
# cane-eval calls this with each test question and judges the response
def agent(question: str) -> str:
    response = chain.invoke({"input": question})
    return response.content

summary = runner.run(suite, agent=agent)

In [ ]:
print(f"Overall score: {summary.overall_score}")
print(f"Pass rate: {summary.pass_rate:.0f}%")
print(f"Passed: {summary.passed} | Warned: {summary.warned} | Failed: {summary.failed}")
print()

# Show each result
for r in summary.results:
    icon = {"pass": "P", "warn": "W", "fail": "F"}[r.status]
    print(f"  [{icon}] {r.score:.0f}  {r.question[:60]}")
    if r.status == "fail":
        print(f"        {r.judge_result.overall_reasoning[:100]}")

## Step 4: Root cause analysis

Go beyond "what failed" to understand "why it failed."
RCA finds patterns across failures and generates actionable recommendations.

In [ ]:
from cane_eval import RootCauseAnalyzer

analyzer = RootCauseAnalyzer(verbose=True)
rca = analyzer.analyze(summary, max_score=80)

print(f"\nSummary: {rca.summary}")
print(f"Top recommendation: {rca.top_recommendation}")
print(f"\n{len(rca.root_causes)} root causes found:")
for rc in rca.root_causes:
    print(f"  [{rc.severity.upper()}] {rc.category}: {rc.title}")
    print(f"    Fix: {rc.recommendation}")

## Step 5: Mine failures into training data

Automatically classify failures, generate improved answers via LLM,
and export as DPO/SFT training pairs for fine-tuning.

In [ ]:
from cane_eval import FailureMiner

miner = FailureMiner(verbose=True)
mined = miner.mine(summary, max_score=80)

print(f"\nMined {mined.total_mined} examples from {mined.total_failures} failures")
print(f"Failure types: {mined.failure_distribution}")

In [ ]:
# Look at a mined example
if mined.examples:
    ex = mined.examples[0]
    print(f"Question: {ex.question}")
    print(f"\nOriginal (bad) answer: {ex.original_answer}")
    print(f"\nImproved answer: {ex.improved_answer}")
    print(f"\nFailure type: {ex.failure_type}")
    print(f"Original score: {ex.original_score}")

In [ ]:
# Export as DPO training pairs
import json

print("DPO training pairs:")
print("=" * 60)
for ex in mined.examples[:3]:
    pair = ex.to_dpo()
    print(json.dumps(pair, indent=2))
    print()

## Step 6: Export training data

Export in your preferred format for fine-tuning.

In [ ]:
from cane_eval import Exporter

exporter = Exporter(summary)

# DPO pairs (prompt + chosen + rejected)
dpo_data = exporter.as_dpo(max_score=80)
print(f"DPO pairs: {len(dpo_data)}")

# SFT examples (prompt + completion)
sft_data = exporter.as_sft(min_score=80)
print(f"SFT examples: {len(sft_data)}")

# OpenAI fine-tuning format
openai_data = exporter.as_openai(min_score=80)
print(f"OpenAI examples: {len(openai_data)}")

# Save to files
# exporter.to_dpo("training_dpo.jsonl", max_score=80)
# exporter.to_sft("training_sft.jsonl", min_score=80)
# exporter.to_openai("training_openai.jsonl", min_score=80)

## The full loop

```
LangChain agent runs
    |
cane-eval scores responses (Claude as judge)
    |
Root cause analysis finds patterns
    |
Failure mining rewrites bad answers
    |
Export as DPO/SFT training data
    |
Fine-tune your model
    |
Agent improves
```

That's the feedback loop. Run it continuously and your agent gets better every cycle.

## Next steps

- **YAML suites**: Define test suites in YAML files for version control
- **CI/CD**: Run `cane-eval run tests.yaml --quiet` in GitHub Actions
- **Regression diffs**: Compare runs with `cane-eval diff v1.json v2.json`
- **Any framework**: Works with LangChain, LlamaIndex, OpenAI, or any Python callable

Full docs: [github.com/colingfly/cane-eval](https://github.com/colingfly/cane-eval)